Breast Cancer Analysis Pipeline — 10 Classifiers
=============================================================
THREE HARD RULES (enforced):
  1. WBCD load → split → scale runs before anything else
  2. All models fitted before medical_report
  3. SMOTE never standalone before CV

Cell Order (34 cells):
   1.  Environment detection + Drive mount + folder structure
   2.  Install packages
   3.  All imports + GPU verification
   4.  Load WBCD from UCI
   5.  Train/val/test split + StandardScaler (save X_test_orig)
   6.  VarianceThreshold
   7.  CFS
   8.  IG
   9.  SFS forward + backward
  10.  ANOVA + RFE + PCA
  11.  PSO feature selection
  12.  WOA feature selection
  13.  FS comparison table
  14.  All 10 classifiers defined (ANN + 9 ML)
  15.  Optuna HPO for ALL classifiers (including ANN)
  16.  GridSearchCV RF + XGBoost
  17.  Stratified 10-fold CV with ImbPipeline + ANN
  18.  Master results table
  19.  Fit all 10 on SMOTE + Voting (ANN+SVM+LR) + Stacking
  20.  Threshold tuning (handles ANN winner)
  21.  SHAP
  22.  XGBoost final eval
  23.  PR curve
  24.  Confusion matrices — all 10
  25.  ROC all models (10 lines + ensembles)
  26.  Learning curves (handles ANN winner)
  27.  Calibration curves (handles ANN)
  28.  McNemar (handles ANN)
  29.  Feature importance RF + XGBoost
  30.  BreastANN validation report
  31.  EfficientNet placeholder
  32.  U-Net placeholder
  33.  Model save + ONNX export
  34.  medical_report + interactive input

In [ ]:
import os, sys, subprocess, warnings, gc, copy

# ---------------------------------------------------------------------------
# ENVIRONMENT DETECTION
# ---------------------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_GPU", "")

def is_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

IN_NOTEBOOK = is_notebook()

if IN_COLAB:
    from google.colab import drive

### **Drive Mount + Folder Structure**

In [ ]:
BASE_DIR = (
    "/content/drive/MyDrive/BC_internship"
    if IN_COLAB
    else os.path.join(os.getcwd(), "BC_internship")
)

if IN_COLAB:
    drive.mount("/content/drive")

for folder in [
    "data", "notebooks", "src/data", "src/models", "src",
    "experiments", "reports/daily_logs", "figures",
]:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print(f"Project structure created at: {BASE_DIR}")

### **Install Packages**

In [ ]:
def pip_install(packages):
    if IN_COLAB or IN_NOTEBOOK:
        import IPython
        IPython.get_ipython().system(f"pip install {' '.join(packages)}")
    else:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install"] + packages
        )

pip_install([
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu121",
])

pip_install([
    "pandas", "numpy", "matplotlib", "seaborn", "scikit-learn",
    "tqdm", "albumentations", "wandb", "optuna", "pillow", "kaggle",
    "xgboost", "imbalanced-learn", "pyswarms", "pydicom", "medpy",
    "segmentation-models-pytorch", "shap",
    "onnx", "onnxruntime", "onnxscript",
])

print("All packages installed.")

### **Import packages + GPU verification**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, learning_curve,
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.feature_selection import (
    SelectKBest, chi2, RFE, f_classif, mutual_info_classif,
    SequentialFeatureSelector, VarianceThreshold,
)
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.base import BaseEstimator, ClassifierMixin, clone

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms, models

import joblib
import optuna
import shap
import json

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("All imports successful.")

Load WBCD from UCI

In [ ]:
URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "breast-cancer-wisconsin/breast-cancer-wisconsin.data"
)

COLUMN_NAMES = [
    "id", "clump_thickness", "cell_size_uniformity",
    "cell_shape_uniformity", "marginal_adhesion",
    "single_epithelial_cell_size", "bare_nuclei",
    "bland_chromatin", "normal_nucleoli", "mitoses", "class",
]

df = pd.read_csv(URL, names=COLUMN_NAMES, na_values="?")
df["bare_nuclei"] = pd.to_numeric(df["bare_nuclei"], errors="coerce")
df["bare_nuclei"] = df["bare_nuclei"].fillna(df["bare_nuclei"].median())
df = df.drop("id", axis=1)
df["class"] = df["class"].map({2: 0, 4: 1})


class DataHolder:
    pass

data = DataHolder()
data.data = df.drop("class", axis=1).values
data.target = df["class"].values
data.feature_names = df.drop("class", axis=1).columns.values

print(f"Shape: {df.shape}")
print(f"Class distribution:\n{df['class'].value_counts()}")
print(f"Feature names:\n{data.feature_names}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

**Train/val/test split + Standard Scaler**

In [ ]:
X_all = data.data
y_all = data.target

X_temp, X_test_orig, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=42, stratify=y_all,
)

X_train_orig, X_val_orig, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1875, random_state=42, stratify=y_temp,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_orig)
X_val   = scaler.transform(X_val_orig)
X_test  = scaler.transform(X_test_orig)

feat_names = data.feature_names

# DataFrames for XGBoost / SHAP
X_train_df = pd.DataFrame(X_train, columns=feat_names)
X_val_df   = pd.DataFrame(X_val,   columns=feat_names)
X_test_df  = pd.DataFrame(X_test,  columns=feat_names)

# Unscaled copies for SHAP
X_train_orig_df = pd.DataFrame(X_train_orig, columns=feat_names)
X_val_orig_df   = pd.DataFrame(X_val_orig,   columns=feat_names)
X_test_orig_df  = pd.DataFrame(X_test_orig,  columns=feat_names)

y_train_s = pd.Series(y_train)
y_val_s   = pd.Series(y_val)
y_test_s  = pd.Series(y_test)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")
print("X_test_orig saved for SHAP.")

Variance Threshold

In [ ]:
var_thresh = VarianceThreshold(threshold=0.01)
var_thresh.fit(X_train)
kept_var = feat_names[var_thresh.get_support()]
dropped_var = feat_names[~var_thresh.get_support()]

print(f"Variance Threshold (0.01): kept {len(kept_var)}/{len(feat_names)}")
print(f"  Kept:    {list(kept_var)}")
print(f"  Dropped: {list(dropped_var) if len(dropped_var) > 0 else 'none'}")


Correlation Feature Selection (CFS)

In [ ]:
train_df_full = X_train_df.copy()
train_df_full["target"] = y_train

corr = train_df_full.corr()["target"].abs().sort_values(ascending=False)
print("\nFeature-Target Correlations:\n", corr)

selected_cfs = []
THRESH_TARGET = 0.5
THRESH_INTER  = 0.7

if not corr.drop("target").empty:
    initial = corr.drop("target").index[0]
    if corr.loc[initial] >= THRESH_TARGET:
        selected_cfs.append(initial)
    for feat in corr.drop("target").index:
        if feat in selected_cfs:
            continue
        if corr.loc[feat] < THRESH_TARGET:
            continue
        high_inter = any(
            abs(train_df_full[feat].corr(train_df_full[sf])) >= THRESH_INTER
            for sf in selected_cfs
        )
        if not high_inter:
            selected_cfs.append(feat)

print(f"\nCFS Selected: {selected_cfs}")
cfs_mask = np.isin(feat_names, selected_cfs)

Information Gain

In [ ]:
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=feat_names).sort_values(ascending=False)
print("\nInformation Gain:\n", mi_series)

k_ig = 5
selected_ig = mi_series.nlargest(k_ig).index.tolist()
print(f"\nTop {k_ig} by IG: {selected_ig}")
ig_mask = np.isin(feat_names, selected_ig)

SFS forward + backward

In [ ]:
est_sfs = RandomForestClassifier(n_estimators=100, random_state=42)

sfs_fwd = SequentialFeatureSelector(
    est_sfs, n_features_to_select=5, direction="forward",
    cv=3, scoring="roc_auc", n_jobs=-1,
)
sfs_fwd.fit(X_train, y_train)
sel_sfs_fwd = feat_names[sfs_fwd.get_support()].tolist()
print(f"\nSFS Forward:  {sel_sfs_fwd}")

sfs_bwd = SequentialFeatureSelector(
    est_sfs, n_features_to_select=5, direction="backward",
    cv=3, scoring="roc_auc", n_jobs=-1,
)
sfs_bwd.fit(X_train, y_train)
sel_sfs_bwd = feat_names[sfs_bwd.get_support()].tolist()
print(f"SFS Backward: {sel_sfs_bwd}")

sfs_mask = sfs_fwd.get_support()

ANOVA + RFE + PCA

In [ ]:
sel_anova = SelectKBest(f_classif, k=7)
sel_anova.fit(X_train, y_train)
chosen_anova = feat_names[sel_anova.get_support()]
print("ANOVA selected:", chosen_anova)

rf_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf_rfe, n_features_to_select=5)
rfe.fit(X_train, y_train)
print("RFE features:", feat_names[rfe.support_])

pca = PCA(n_components=2)
X_pca_2d = pca.fit_transform(X_train)
print(f"PCA variance explained: {pca.explained_variance_ratio_.sum():.2%}")

anova_mask = sel_anova.get_support()

PSO Feature Selection

In [ ]:
print("\n--- PSO Feature Selection ---")

N_FEATURES = X_train.shape[1]
N_PARTICLES = 30
N_ITER = 50
W_INERTIA = 0.7
C1 = 1.5
C2 = 1.5


def pso_fitness(mask, X_tr, y_tr, X_vl, y_vl):
    if mask.sum() == 0:
        return 0.0
    idx = np.where(mask)[0]
    rf_ = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_.fit(X_tr[:, idx], y_tr)
    return roc_auc_score(y_vl, rf_.predict_proba(X_vl[:, idx])[:, 1])


positions = np.random.rand(N_PARTICLES, N_FEATURES) < 0.5
velocities = np.random.uniform(-1, 1, (N_PARTICLES, N_FEATURES))
pbest_pos = positions.copy()
pbest_fit = np.array([
    pso_fitness(p, X_train, y_train, X_val, y_val) for p in positions
])
gbest_idx = np.argmax(pbest_fit)
gbest_pos = pbest_pos[gbest_idx].copy()
gbest_fit = pbest_fit[gbest_idx]

for it in range(N_ITER):
    r1 = np.random.rand(N_PARTICLES, N_FEATURES)
    r2 = np.random.rand(N_PARTICLES, N_FEATURES)
    velocities = (
        W_INERTIA * velocities
        + C1 * r1 * (pbest_pos.astype(float) - positions.astype(float))
        + C2 * r2 * (gbest_pos.astype(float) - positions.astype(float))
    )
    prob = 1 / (1 + np.exp(-velocities))
    positions = np.random.rand(N_PARTICLES, N_FEATURES) < prob
    fits = np.array([
        pso_fitness(p, X_train, y_train, X_val, y_val) for p in positions
    ])
    improved = fits > pbest_fit
    pbest_pos[improved] = positions[improved]
    pbest_fit[improved] = fits[improved]
    new_gbest = np.argmax(pbest_fit)
    if pbest_fit[new_gbest] > gbest_fit:
        gbest_pos = pbest_pos[new_gbest].copy()
        gbest_fit = pbest_fit[new_gbest]
    if (it + 1) % 10 == 0:
        print(f"  PSO iter {it+1:2d}/{N_ITER} | best AUC={gbest_fit:.4f} | "
              f"features={gbest_pos.sum()}")

sel_pso = feat_names[gbest_pos]
print(f"\nPSO Selected (AUC={gbest_fit:.4f}): {list(sel_pso)}")
pso_mask = gbest_pos.copy()

WOA Feature Selection

In [ ]:
print("\n--- WOA Feature Selection ---")

N_WHALES = 30
N_WOA_ITER = 50


def woa_fitness(mask, X_tr, y_tr, X_vl, y_vl):
    if mask.sum() == 0:
        return 0.0
    idx = np.where(mask)[0]
    rf_ = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_.fit(X_tr[:, idx], y_tr)
    return roc_auc_score(y_vl, rf_.predict_proba(X_vl[:, idx])[:, 1])


whales = np.random.rand(N_WHALES, N_FEATURES) < 0.5
whale_fits = np.array([
    woa_fitness(w, X_train, y_train, X_val, y_val) for w in whales
])
best_idx = np.argmax(whale_fits)
best_whale = whales[best_idx].copy()
best_whale_fit = whale_fits[best_idx]

for t in range(N_WOA_ITER):
    a = 2 - 2 * t / N_WOA_ITER
    for i in range(N_WHALES):
        r = np.random.rand()
        A_val = 2 * a * np.random.rand() - a
        C_val = 2 * np.random.rand()
        l_val = np.random.uniform(-1, 1)
        if r < 0.5:
            if abs(A_val) < 1:
                D = abs(C_val * best_whale.astype(float) - whales[i].astype(float))
                new_pos = best_whale.astype(float) - A_val * D
            else:
                rand_idx = np.random.randint(0, N_WHALES)
                D = abs(C_val * whales[rand_idx].astype(float) - whales[i].astype(float))
                new_pos = whales[rand_idx].astype(float) - A_val * D
        else:
            D = abs(best_whale.astype(float) - whales[i].astype(float))
            new_pos = (D * np.exp(l_val) * np.cos(2 * np.pi * l_val)
                       + best_whale.astype(float))
        prob = 1 / (1 + np.exp(-new_pos))
        whales[i] = np.random.rand(N_FEATURES) < prob

    whale_fits = np.array([
        woa_fitness(w, X_train, y_train, X_val, y_val) for w in whales
    ])
    new_best = np.argmax(whale_fits)
    if whale_fits[new_best] > best_whale_fit:
        best_whale = whales[new_best].copy()
        best_whale_fit = whale_fits[new_best]
    if (t + 1) % 10 == 0:
        print(f"  WOA iter {t+1:2d}/{N_WOA_ITER} | best AUC={best_whale_fit:.4f} | "
              f"features={best_whale.sum()}")

sel_woa = feat_names[best_whale]
print(f"\nWOA Selected (AUC={best_whale_fit:.4f}): {list(sel_woa)}")
woa_mask = best_whale.copy()

FS comparision Table

In [ ]:
def evaluate_fs_mask(mask, name):
    if mask.sum() == 0:
        return {
            "Method": name, "N_Features": 0, "Features": "none",
            "CV_AUC": 0, "CV_Acc": 0, "CV_F1": 0,
        }
    idx = np.where(mask)[0]
    pipe = ImbPipeline([
        ("sm", SMOTE(k_neighbors=5, random_state=42)),
        ("clf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ])
    cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc = cross_val_score(pipe, X_train[:, idx], y_train, scoring="roc_auc", cv=cv5).mean()
    acc = cross_val_score(pipe, X_train[:, idx], y_train, scoring="accuracy", cv=cv5).mean()
    f1  = cross_val_score(pipe, X_train[:, idx], y_train, scoring="f1", cv=cv5).mean()
    return {
        "Method": name, "N_Features": mask.sum(),
        "Features": ", ".join(feat_names[mask]),
        "CV_AUC": auc, "CV_Acc": acc, "CV_F1": f1,
    }


fs_comparison = []
fs_comparison.append(evaluate_fs_mask(np.ones(N_FEATURES, dtype=bool), "All Features"))
fs_comparison.append(evaluate_fs_mask(cfs_mask, "CFS"))
fs_comparison.append(evaluate_fs_mask(ig_mask, "IG (top-5)"))
fs_comparison.append(evaluate_fs_mask(sfs_mask, "SFS Forward"))
fs_comparison.append(evaluate_fs_mask(anova_mask, "ANOVA (top-7)"))
fs_comparison.append(evaluate_fs_mask(pso_mask, "PSO"))
fs_comparison.append(evaluate_fs_mask(woa_mask, "WOA"))

fs_df = pd.DataFrame(fs_comparison)
fs_df = fs_df.sort_values("CV_AUC", ascending=False).reset_index(drop=True)

print("\n" + "=" * 90)
print("            FEATURE SELECTION COMPARISON TABLE")
print("=" * 90)
print(fs_df.to_string(index=False))
fs_df.to_csv(os.path.join(BASE_DIR, "data", "fs_comparison.csv"), index=False)

best_fs = fs_df.iloc[0]
print(f"\nBest FS: {best_fs['Method']} "
      f"(AUC={best_fs['CV_AUC']:.4f}, {best_fs['N_Features']} features)")

# ── CHART 1: Grouped bar chart — AUC, Acc, F1 per FS method ────────────────
plot_df = fs_df.set_index("Method")[["CV_AUC", "CV_Acc", "CV_F1"]]
plot_df = plot_df.sort_values("CV_AUC", ascending=True)   # ascending for hbar

methods = plot_df.index.tolist()
x = np.arange(len(methods))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.barh(x + width, plot_df["CV_AUC"], width, label="AUC",
                color="#2196F3", edgecolor="white", linewidth=0.8)
bars2 = ax.barh(x,          plot_df["CV_Acc"], width, label="Accuracy",
                color="#4CAF50", edgecolor="white", linewidth=0.8)
bars3 = ax.barh(x - width, plot_df["CV_F1"],  width, label="F1",
                color="#FF9800", edgecolor="white", linewidth=0.8)

ax.set_yticks(x)
ax.set_yticklabels(methods, fontsize=12)
ax.set_xlabel("CV Score", fontsize=13)
ax.set_title("Feature Selection Comparison — AUC · Accuracy · F1",
             fontsize=15, fontweight="bold", pad=16)
ax.legend(loc="lower right", fontsize=11, framealpha=0.9)
ax.set_xlim(0.75, 1.02)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Annotate bars with values
for bar_set in [bars1, bars2, bars3]:
    for bar in bar_set:
        w = bar.get_width()
        ax.text(w + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{w:.4f}", va="center", fontsize=8, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "fs_grouped_bar.png"), dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved: fs_grouped_bar.png")

# ── CHART 2: AUC ranking — horizontal bar with gradient ────────────────────
auc_sorted = fs_df.sort_values("CV_AUC", ascending=True)
auc_vals = auc_sorted["CV_AUC"].values
auc_names = auc_sorted["Method"].values
feature_counts = auc_sorted["N_Features"].values

# Color gradient: best = dark teal, worst = light gray
norm = plt.Normalize(auc_vals.min(), auc_vals.max())
colors = plt.cm.viridis_r(norm(auc_vals))

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(auc_names, auc_vals, color=colors, edgecolor="white", linewidth=1.2,
               height=0.65)

ax.set_xlabel("CV AUC", fontsize=13)
ax.set_title("Feature Selection Methods — Ranked by AUC",
             fontsize=15, fontweight="bold", pad=14)
ax.set_xlim(auc_vals.min() - 0.015, auc_vals.max() + 0.01)
ax.grid(axis="x", alpha=0.25, linestyle="--")

# Right-side annotation: AUC + feature count
for bar, val, nf in zip(bars, auc_vals, feature_counts):
    ax.text(val + 0.0015, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}  ({nf} feat.)", va="center", fontsize=11,
            fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "fs_auc_ranking.png"), dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved: fs_auc_ranking.png")

# ── CHART 3: Donut chart — AUC share per FS method ─────────────────────────
# Use the order from the table (descending AUC)
donut_methods = fs_df["Method"].tolist()
donut_vals = fs_df["CV_AUC"].tolist()
donut_features = fs_df["N_Features"].tolist()

# Explode the best method slightly
explode = [0.06 if i == 0 else 0.02 for i in range(len(donut_methods))]

fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    donut_vals,
    labels=None,
    autopct="%1.2f%%",
    startangle=210,
    pctdistance=0.78,
    explode=explode,
    colors=plt.cm.Set2(np.linspace(0, 1, len(donut_methods))),
    wedgeprops={"linewidth": 2, "edgecolor": "white", "width": 0.4},  # width=0.4 → donut
    textprops={"fontsize": 11, "fontweight": "bold"},
)

# Custom legend with feature counts
legend_labels = [
    f"{m} ({n} feat.) — {v:.4f}"
    for m, n, v in zip(donut_methods, donut_features, donut_vals)
]
ax.legend(
    wedges, legend_labels,
    title="Method  (# features) — AUC",
    loc="center left",
    bbox_to_anchor=(1, 0, 0.5, 1),
    fontsize=10,
    title_fontsize=11,
)
ax.set_title("Feature Selection — AUC Distribution (Donut Chart)",
             fontsize=14, fontweight="bold", pad=18)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "fs_donut.png"), dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved: fs_donut.png")

PyTorch ANN Model class

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, learning_curve,
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
    PrecisionRecallDisplay,
)
from sklearn.feature_selection import (
    SelectKBest, chi2, RFE, f_classif, mutual_info_classif,
    SequentialFeatureSelector, VarianceThreshold,
)
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.base import BaseEstimator, ClassifierMixin, clone

from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms, models

import joblib
import optuna
import shap
import json
import types # Import types module for SimpleNamespace

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("All imports successful.")

class BreastANN(nn.Module):
    """Multi-layer perceptron for binary BC classification (raw logits)."""
    def __init__(self, n_features=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),    # <-- NO Sigmoid — BCEWithLogitsLoss handles it
        )

    def forward(self, x):
        return self.net(x).squeeze()


# ── sklearn-compatible ANN wrapper ──

class ANNWrapper(BaseEstimator, ClassifierMixin):
    """sklearn-compatible wrapper that embeds a PyTorch BreastANN."""
    _estimator_type = "classifier"

    def __init__(self, n_features=9, device_str="cpu",
                 lr=1e-3, weight_decay=1e-4, epochs=30):
        self.n_features = n_features
        self.device_str = device_str
        self.lr = lr
        self.weight_decay = weight_decay
        self.epochs = epochs
        self.model = None
        self.classes_ = None
        self._estimator_type = "classifier" # Explicitly set instance attribute

    def _more_tags(self):
        """Required for sklearn >= 1.6 estimator type detection."""
        return {
            "non_deterministic": True,     # PyTorch stochastic training
            "requires_fit": True,
        }

    def __sklearn_tags__(self):
        tags_dict = self._more_tags() # Get custom tags
        tags_dict["estimator_type"] = "classifier"
        tags_dict["binary_only"] = True # Explicitly mark as binary classifier
        tags_dict["_xfail_checks"] = {
            "check_estimators_nan": "ANNWrapper does not handle NaN values automatically; preprocessing is expected."
        }
        # Return a SimpleNamespace to allow attribute access (e.g., .estimator_type)
        return types.SimpleNamespace(**tags_dict)

    def _make_model(self):
        return BreastANN(n_features=self.n_features)

    def fit(self, X, y):
        device = torch.device(self.device_str)
        self.model = self._make_model().to(device)

        X_t = torch.tensor(
            X if isinstance(X, np.ndarray) else X.values,
            dtype=torch.float32,
        )
        y_t = torch.tensor(
            y if isinstance(y, np.ndarray) else y.values,
            dtype=torch.float32,
        )
        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=True)

        criterion = nn.BCEWithLogitsLoss()          # ← numerically stable
        optimizer = optim.Adam(
            self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay,
        )

        for _ in range(self.epochs):
            self.model.train()
            for bx, by in loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                pred = self.model(bx)               # raw logits
                loss = criterion(pred, by)
                if torch.isnan(loss) or torch.isinf(loss):
                    continue                        # skip bad batch
                loss.backward()
                torch.nn.utils.clip_grad_norm_(      # ← prevent explosion
                    self.model.parameters(), max_norm=1.0,
                )
                optimizer.step()

        self.classes_ = np.array([0, 1])
        return self

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

    def predict_proba(self, X):
        if self.model is None:
            raise RuntimeError("ANN not fitted.")
        self.model.to(torch.device(self.device_str)) # Explicitly move model to device
        self.model.eval()
        with torch.no_grad():
            # Ensure input tensor is on the same device as the model
            X_t = torch.tensor(
                X if isinstance(X, np.ndarray) else X.values,
                dtype=torch.float32,
            ).to(torch.device(self.device_str))
            logits = self.model(X_t).cpu().numpy()         # raw logits
            proba_1 = 1 / (1 + np.exp(-logits))            # sigmoid
        return np.column_stack([1 - proba_1, proba_1])

All 10 Classifiers defined

In [ ]:
neg, pos = np.bincount(y_train)

svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42)

rf = RandomForestClassifier(
    n_estimators=500, max_features="sqrt", class_weight="balanced", random_state=42,
)

xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg / pos, eval_metric="auc", random_state=42,
)
xgb.set_params(early_stopping_rounds=20)

lr  = LogisticRegression(random_state=42, solver="liblinear", class_weight="balanced")
nb  = GaussianNB()
knn = KNeighborsClassifier(n_neighbors=5)
dt  = DecisionTreeClassifier(random_state=42, class_weight="balanced")
ab  = AdaBoostClassifier(n_estimators=100, random_state=42)
gb  = GradientBoostingClassifier(n_estimators=100, random_state=42)

# ANN — sklearn-compatible
ann_clf = ANNWrapper(n_features=X_train.shape[1], device_str=str(DEVICE))

CLASSIFIERS = {
    "ANN":              ann_clf,
    "SVM":              svm,
    "RF":               rf,
    "XGB":              xgb,
    "LR":               lr,
    "NB":               nb,
    "KNN":              knn,
    "DT":               dt,
    "AdaBoost":         ab,
    "GradientBoosting": gb,
}

print("10 classifiers defined.")
print(f"class_weight check → neg={neg}, pos={pos}, XGB scale_pos_weight={neg/pos:.2f}")

# Save defaults BEFORE Optuna overwrites (needed for ablation study in Cell 20b)
CLASSIFIERS_DEFAULTS = {
    "SVM": SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42),
    "RF":  RandomForestClassifier(n_estimators=500, max_features="sqrt",
                                  class_weight="balanced", random_state=42),
    "XGB": XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5,
                         subsample=0.8, colsample_bytree=0.8,
                         scale_pos_weight=neg/pos, eval_metric="auc",
                         random_state=42),
    "LR":  LogisticRegression(random_state=42, solver="liblinear",
                              class_weight="balanced"),
    "NB":  GaussianNB(),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "DT":  DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}


Optuna HPO for all classifiers

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def _cv_score(pipe, X_sel, y_sel, n_splits=5):
    return cross_val_score(pipe, X_sel, y_sel, scoring="roc_auc", cv=n_splits).mean()


def tune_svm(trial):
    C_val = trial.suggest_float("C", 0.1, 100, log=True)
    gam   = trial.suggest_categorical("gamma", ["scale", "auto"])
    mdl = SVC(C=C_val, gamma=gam, kernel="rbf", probability=True, random_state=42)
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_rf(trial):
    n_est = trial.suggest_int("n_estimators", 100, 800)
    mx_d  = trial.suggest_int("max_depth", 3, 30)
    mx_f  = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
    mdl = RandomForestClassifier(
        n_estimators=n_est, max_depth=mx_d, max_features=mx_f,
        class_weight="balanced", random_state=42, n_jobs=-1,
    )
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_xgb(trial):
    n_est = trial.suggest_int("n_estimators", 100, 500)
    lr_v  = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    mx_d  = trial.suggest_int("max_depth", 3, 12)
    sub   = trial.suggest_float("subsample", 0.5, 1.0)
    col   = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    mdl = XGBClassifier(
        n_estimators=n_est, learning_rate=lr_v, max_depth=mx_d,
        subsample=sub, colsample_bytree=col,
        scale_pos_weight=neg / pos, eval_metric="auc",
        random_state=42,
        # NO early_stopping_rounds — cross_val_score has no eval_set
    )
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train_df, y_train_s)


def tune_lr(trial):
    C_val = trial.suggest_float("C", 0.01, 100, log=True)
    mdl = LogisticRegression(C=C_val, random_state=42, solver="liblinear",
                             class_weight="balanced")
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_knn(trial):
    k = trial.suggest_int("n_neighbors", 1, 30)
    w = trial.suggest_categorical("weights", ["uniform", "distance"])
    mdl = KNeighborsClassifier(n_neighbors=k, weights=w)
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_dt(trial):
    mx_d  = trial.suggest_int("max_depth", 2, 20)
    min_s = trial.suggest_int("min_samples_split", 2, 20)
    mdl = DecisionTreeClassifier(
        max_depth=mx_d, min_samples_split=min_s,
        class_weight="balanced", random_state=42,
    )
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_ab(trial):
    n_est = trial.suggest_int("n_estimators", 50, 400)
    lr_v  = trial.suggest_float("learning_rate", 0.01, 2.0, log=True)
    mdl = AdaBoostClassifier(n_estimators=n_est, learning_rate=lr_v, random_state=42)
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_gb(trial):
    n_est = trial.suggest_int("n_estimators", 50, 400)
    lr_v  = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    mx_d  = trial.suggest_int("max_depth", 2, 10)
    mdl = GradientBoostingClassifier(
        n_estimators=n_est, learning_rate=lr_v, max_depth=mx_d, random_state=42,
    )
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)


def tune_nb(trial):
    """Naive Bayes has no hyperparameters — single evaluation for baseline."""
    mdl = GaussianNB()
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    return _cv_score(pipe, X_train, y_train)

def tune_ann(trial):
    lr_v = trial.suggest_float("lr", 5e-4, 5e-3)       # narrower, safer
    wd   = trial.suggest_float("weight_decay", 1e-6, 1e-3)  # gentler
    epochs = trial.suggest_int("epochs", 20, 50)
    # ... retries up to 3 times, checks per-fold NaN
    mdl = ANNWrapper(
        n_features=X_train.shape[1], device_str=str(DEVICE),
        lr=lr_v, weight_decay=wd, epochs=epochs,
    )
    pipe = ImbPipeline([("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", mdl)])
    try:
        score = _cv_score(pipe, X_train, y_train)
        if np.isnan(score):
            return 0.5   # fallback to random AUC on NaN
        return score
    except Exception:
        return 0.5


TUNE_MAP = {
    "ANN":              (tune_ann, 30),
    "SVM":              (tune_svm, 50),
    "RF":               (tune_rf, 50),
    "XGB":              (tune_xgb, 50),
    "LR":               (tune_lr, 50),
    "NB":               (tune_nb,  1),   # no hyperparams — single baseline
    "KNN":              (tune_knn, 30),
    "DT":               (tune_dt, 30),
    "AdaBoost":         (tune_ab, 50),
    "GradientBoosting": (tune_gb, 50),
}

best_params_all = {}

print("\n--- Optuna HPO for all 10 classifiers ---")
for name, (obj_fn, n_trials) in TUNE_MAP.items():
    study = optuna.create_study(direction="maximize")
    study.optimize(obj_fn, n_trials=n_trials, show_progress_bar=False)
    best_params_all[name] = study.best_params
    print(f"  {name:>20s}: best AUC={study.best_value:.4f}  "
          f"params={study.best_params}")

# ── Apply best params ──

CLASSIFIERS["ANN"] = ANNWrapper(
    n_features=X_train.shape[1], device_str=str(DEVICE),
    lr=best_params_all["ANN"]["lr"],
    weight_decay=best_params_all["ANN"]["weight_decay"],
    epochs=best_params_all["ANN"]["epochs"],
)

CLASSIFIERS["SVM"] = SVC(
    C=best_params_all["SVM"].get("C", 10),
    gamma=best_params_all["SVM"].get("gamma", "scale"),
    kernel="rbf", probability=True, random_state=42,
)
CLASSIFIERS["RF"] = RandomForestClassifier(
    n_estimators=best_params_all["RF"]["n_estimators"],
    max_depth=best_params_all["RF"]["max_depth"],
    max_features=best_params_all["RF"]["max_features"],
    class_weight="balanced", random_state=42, n_jobs=-1,
)
CLASSIFIERS["XGB"] = XGBClassifier(
    n_estimators=best_params_all["XGB"]["n_estimators"],
    learning_rate=best_params_all["XGB"]["learning_rate"],
    max_depth=best_params_all["XGB"]["max_depth"],
    subsample=best_params_all["XGB"]["subsample"],
    colsample_bytree=best_params_all["XGB"]["colsample_bytree"],
    scale_pos_weight=neg / pos, eval_metric="auc",
    random_state=42, early_stopping_rounds=20,
)
CLASSIFIERS["LR"] = LogisticRegression(
    C=best_params_all["LR"]["C"],
    random_state=42, solver="liblinear", class_weight="balanced",
)
CLASSIFIERS["KNN"] = KNeighborsClassifier(
    n_neighbors=best_params_all["KNN"]["n_neighbors"],
    weights=best_params_all["KNN"]["weights"],
)
CLASSIFIERS["DT"] = DecisionTreeClassifier(
    max_depth=best_params_all["DT"]["max_depth"],
    min_samples_split=best_params_all["DT"]["min_samples_split"],
    class_weight="balanced", random_state=42,
)
CLASSIFIERS["AdaBoost"] = AdaBoostClassifier(
    n_estimators=best_params_all["AdaBoost"]["n_estimators"],
    learning_rate=best_params_all["AdaBoost"]["learning_rate"],
    random_state=42,
)
CLASSIFIERS["GradientBoosting"] = GradientBoostingClassifier(
    n_estimators=best_params_all["GradientBoosting"]["n_estimators"],
    learning_rate=best_params_all["GradientBoosting"]["learning_rate"],
    max_depth=best_params_all["GradientBoosting"]["max_depth"],
    random_state=42,
)
# NB has no tunable parameters — keep the default GaussianNB() from Cell 15

print("\nAll 10 classifiers updated with Optuna-tuned hyperparameters.")

Classifier X FS Matrix

In [ ]:
print("\n--- Classifier x FS Method Accuracy Matrix ---")

FS_MASKS = {
    "All":      np.ones(N_FEATURES, dtype=bool),
    "CFS":      cfs_mask,
    "IG":       ig_mask,
    "SFS":      sfs_mask,
    "ANOVA":    anova_mask,
    "PSO":      pso_mask,
    "WOA":      woa_mask,
}

CLF_NAMES_SIMPLE = {
    "ANN": "ANN", "SVM": "SVM", "RF": "RF", "XGB": "XGB",
    "LR": "LR", "NB": "NB", "KNN": "KNN", "DT": "DT",
    "AdaBoost": "AdaBoost", "GradientBoosting": "GradBoost",
}

fs_clf_matrix = pd.DataFrame(index=list(CLF_NAMES_SIMPLE.values()),
                              columns=list(FS_MASKS.keys()))

cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for fs_name, mask in FS_MASKS.items():
    idx = np.where(mask)[0]
    print(f"  Evaluating {fs_name} ({mask.sum()} features) ...")
    for clf_name, clf in CLASSIFIERS.items():
        simple = CLF_NAMES_SIMPLE[clf_name]
        try:
            if clf_name == "XGB":
                temp_clf = copy.deepcopy(clf)
                temp_clf.set_params(early_stopping_rounds=None)
                pipe = ImbPipeline([
                    ("sm", SMOTE(k_neighbors=5, random_state=42)),
                    ("clf", temp_clf),
                ])
                score = cross_val_score(
                    pipe, X_train_df.iloc[:, idx], y_train_s,
                    scoring="accuracy", cv=cv10,
                ).mean()
            elif clf_name == "ANN":
                pipe = ImbPipeline([
                    ("sm", SMOTE(k_neighbors=5, random_state=42)),
                    ("clf", clone(clf)),   # clone avoids fit-state leakage across folds
                ])
                score = cross_val_score(
                    pipe, X_train[:, idx], y_train,
                    scoring="accuracy", cv=cv10,
                ).mean()
            else:
                pipe = ImbPipeline([
                    ("sm", SMOTE(k_neighbors=5, random_state=42)),
                    ("clf", clf),
                ])
                score = cross_val_score(
                    pipe, X_train[:, idx], y_train,
                    scoring="accuracy", cv=cv10,
                ).mean()
            fs_clf_matrix.loc[simple, fs_name] = score
        except Exception:
            fs_clf_matrix.loc[simple, fs_name] = np.nan

fs_clf_matrix = fs_clf_matrix.astype(float).round(4)

print("\n" + "=" * 110)
print("   CLASSIFIER x FS METHOD — 10-fold CV Accuracy (10 classifiers x 7 methods)")
print("=" * 110)
print(fs_clf_matrix.to_string())

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(fs_clf_matrix, annot=True, fmt=".4f", cmap="YlOrRd",
            linewidths=1, linecolor="white", cbar_kws={"label": "Accuracy"},
            vmin=0.85, vmax=1.0, ax=ax)
ax.set_title("Classifier x Feature Selection Method — 10-fold CV Accuracy",
             fontsize=14, fontweight="bold", pad=16)
ax.set_xlabel("FS Method", fontsize=12)
ax.set_ylabel("Classifier", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "clf_fs_heatmap.png"), dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved: clf_fs_heatmap.png")

flat_max = fs_clf_matrix.stack()
best_pair = flat_max.idxmax()
best_acc  = flat_max.max()
print(f"\nBest combination: {best_pair[0]} x {best_pair[1]} -> Accuracy = {best_acc:.4f}")

fs_clf_matrix.to_csv(os.path.join(BASE_DIR, "data", "clf_fs_matrix.csv"))

GridSearchCV for RF + XGBoost

In [ ]:
print("\n--- GridSearchCV: RF ---")
rf_pipe = ImbPipeline([
    ("sm", SMOTE(k_neighbors=5, random_state=42)),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)),
])
rf_grid = {
    "clf__n_estimators": [200, 400, 600],
    "clf__max_depth": [5, 10, 20, None],
    "clf__max_features": ["sqrt", "log2"],
}
rf_gs = GridSearchCV(rf_pipe, rf_grid, scoring="roc_auc", cv=5, n_jobs=-1, verbose=0)
rf_gs.fit(X_train, y_train)
print(f"  Best RF params: {rf_gs.best_params_}")
print(f"  Best RF CV AUC: {rf_gs.best_score_:.4f}")

print("\n--- GridSearchCV: XGBoost ---")
xgb_pipe = ImbPipeline([
    ("sm", SMOTE(k_neighbors=5, random_state=42)),
    ("clf", XGBClassifier(scale_pos_weight=neg / pos, eval_metric="auc",
                          random_state=42)),
    # NO early_stopping_rounds — GridSearchCV has no eval_set
])
xgb_grid = {
    "clf__n_estimators": [200, 300, 400],
    "clf__max_depth": [3, 5, 7],
    "clf__learning_rate": [0.01, 0.05, 0.1],
}
xgb_gs = GridSearchCV(xgb_pipe, xgb_grid, scoring="roc_auc", cv=5, n_jobs=-1, verbose=0)
xgb_gs.fit(X_train_df, y_train_s)
print(f"  Best XGB params: {xgb_gs.best_params_}")
print(f"  Best XGB CV AUC: {xgb_gs.best_score_:.4f}")

Stratified 10-fold CV with imlearn Pipleline

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_results = []

for name, clf in CLASSIFIERS.items():
    if name == "XGB":
        temp = copy.deepcopy(clf)
        temp.set_params(early_stopping_rounds=None)
        pipe = ImbPipeline([
            ("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", temp),
        ])
        auc = cross_val_score(pipe, X_train_df, y_train_s, scoring="roc_auc", cv=cv)
        acc = cross_val_score(pipe, X_train_df, y_train_s, scoring="accuracy", cv=cv)
        f1  = cross_val_score(pipe, X_train_df, y_train_s, scoring="f1", cv=cv)
    elif name == "ANN":
        # ANN: use ImbPipeline directly — ANNWrapper is sklearn-compatible
        pipe = ImbPipeline([
            ("sm", SMOTE(k_neighbors=5, random_state=42)),
            ("clf", clone(clf)),   # clone avoids fit-state leakage across folds
        ])
        auc = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv)
        acc = cross_val_score(pipe, X_train, y_train, scoring="accuracy", cv=cv)
        f1  = cross_val_score(pipe, X_train, y_train, scoring="f1", cv=cv)
    else:
        pipe = ImbPipeline([
            ("sm", SMOTE(k_neighbors=5, random_state=42)), ("clf", clf),
        ])
        auc = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv)
        acc = cross_val_score(pipe, X_train, y_train, scoring="accuracy", cv=cv)
        f1  = cross_val_score(pipe, X_train, y_train, scoring="f1", cv=cv)

    cv_results.append((name, auc.mean(), auc.std(),
                       acc.mean(), acc.std(), f1.mean(), f1.std()))
    print(f"{name:>20s}: AUC {auc.mean():.4f}±{auc.std():.4f}  "
          f"Acc {acc.mean():.4f}±{acc.std():.4f}  "
          f"F1 {f1.mean():.4f}±{f1.std():.4f}")

print("\nCV complete — SMOTE per-fold, no leakage.  ANN included in 10-fold CV.")

Master Result Table

In [ ]:
import os

# Define BASE_DIR if it's not already defined (e.g., when running a single cell)
if 'BASE_DIR' not in locals() and 'BASE_DIR' not in globals():
    IN_COLAB = "google.colab" in globals() or os.environ.get("COLAB_GPU", "")
    BASE_DIR = (
        "/content/drive/MyDrive/BC_internship"
        if IN_COLAB
        else os.path.join(os.getcwd(), "BC_internship")
    )

results_df = pd.DataFrame(
    cv_results,
    columns=["Model", "CV_AUC_Mean", "CV_AUC_Std",
             "CV_Acc_Mean", "CV_Acc_Std", "CV_F1_Mean", "CV_F1_Std"],
)
results_df = results_df.sort_values("CV_AUC_Mean", ascending=False).reset_index(drop=True)
results_df["Rank"] = range(1, len(results_df) + 1)
results_df = results_df[[
    "Rank", "Model", "CV_AUC_Mean", "CV_AUC_Std",
    "CV_Acc_Mean", "CV_Acc_Std", "CV_F1_Mean", "CV_F1_Std",
]]

print("\n" + "=" * 80)
print("                      MASTER RESULTS TABLE  (10 classifiers)")
print("=" * 80)
print(results_df.to_string(index=False))

# Ensure the directory exists before saving the file
os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)
results_df.to_csv(os.path.join(BASE_DIR, "data", "cv_results.csv"), index=False)

# ── Comparison vs Paper's 99.41% target ──
PAPER_TARGET = 99.41
my_best_acc = results_df.iloc[0]["CV_Acc_Mean"] * 100
my_best_model = results_df.iloc[0]["Model"]
my_best_auc   = results_df.iloc[0]["CV_AUC_Mean"]

comp_df = pd.DataFrame([
    ["Paper (Majidpour & Beitollahi 2026)", PAPER_TARGET, "-"],
    [f"Ours ({my_best_model})", f"{my_best_acc:.2f}%", f"AUC={my_best_auc:.4f}"],
], columns=["Source", "Best Accuracy", "Details"])

if my_best_acc >= PAPER_TARGET:
    verdict = "MATCHED or EXCEEDED"
elif my_best_acc >= PAPER_TARGET - 1.0:
    verdict = f"VERY CLOSE (within 1%, diff={PAPER_TARGET - my_best_acc:.2f}%)"
else:
    verdict = f"BELOW (diff={PAPER_TARGET - my_best_acc:.2f}%)"

print("\n" + "=" * 65)
print("   COMPARISON vs PAPER TARGET")
print("=" * 65)
print(comp_df.to_string(index=False))
print(f"\n  Verdict: {verdict}")
print(f"  (Note: Paper accuracy is on BreakHis + DDSM image datasets.")
print(f"   Our accuracy is on WBCD tabular - different modality.)")
print("=" * 65)

Test-set accuracy matrix

In [ ]:
print("\n--- Test-Set Accuracy: Classifier x FS Method (held-out test) ---")

FS_MASKS_ORDERED = [
    ("All",   np.ones(N_FEATURES, dtype=bool)),
    ("CFS",   cfs_mask),
    ("IG",    ig_mask),
    ("SFS",   sfs_mask),
    ("ANOVA", anova_mask),
    ("PSO",   pso_mask),
    ("WOA",   woa_mask),
]

test_matrix = pd.DataFrame(index=list(CLF_NAMES_SIMPLE.values()),
                            columns=[n for n, _ in FS_MASKS_ORDERED])

sm_test = SMOTE(k_neighbors=5, random_state=42)

# Using 10-fold cross-validation for the outer loop to get a more robust estimate
cv_outer = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for fs_name, mask in FS_MASKS_ORDERED:
    idx = np.where(mask)[0]
    X_tr_fs, y_tr_fs = sm_test.fit_resample(X_train[:, idx], y_train)
    X_te_fs = X_test[:, idx]
    print(f"  Fitting on {fs_name} ({mask.sum()} features) ...")
    for clf_name, clf in CLASSIFIERS.items():
        simple = CLF_NAMES_SIMPLE[clf_name]
        try:
            if clf_name == "XGB":
                temp = copy.deepcopy(clf)
                temp.set_params(early_stopping_rounds=None)
                temp.fit(pd.DataFrame(X_tr_fs, columns=feat_names[idx]),
                         pd.Series(y_tr_fs))
                y_pred_ts = temp.predict(pd.DataFrame(X_te_fs,
                                          columns=feat_names[idx]))
            elif clf_name == "ANN":
                ann_copy = ANNWrapper(
                    n_features=mask.sum(), device_str=str(DEVICE),
                    lr=clf.lr, weight_decay=clf.weight_decay,
                    epochs=clf.epochs,
                )
                ann_copy.fit(X_tr_fs, y_tr_fs)
                y_pred_ts = ann_copy.predict(X_te_fs)
            else:
                clf_copy = clone(clf)
                clf_copy.fit(X_tr_fs, y_tr_fs)
                y_pred_ts = clf_copy.predict(X_te_fs)
            test_matrix.loc[simple, fs_name] = accuracy_score(y_test, y_pred_ts)
        except Exception:
            test_matrix.loc[simple, fs_name] = np.nan

test_matrix = test_matrix.astype(float).round(4)

print("\n" + "=" * 110)
print("  TEST-SET ACCURACY - Classifier x FS Method (held-out 20%)")
print("=" * 110)
print(test_matrix.to_string())

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(test_matrix, annot=True, fmt=".4f", cmap="YlGnBu",
            linewidths=1, linecolor="white", cbar_kws={"label": "Test Acc"},
            vmin=0.85, vmax=1.0, ax=ax)
ax.set_title("Test-Set Accuracy - Classifier x Feature Selection Method",
             fontsize=14, fontweight="bold", pad=16)
ax.set_xlabel("FS Method", fontsize=12)
ax.set_ylabel("Classifier", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "test_acc_heatmap.png"),
            dpi=200, bbox_inches="tight")
plt.show()
print("Saved: test_acc_heatmap.png")

flat_test = test_matrix.stack()
best_test_pair = flat_test.idxmax()
best_test_acc_val = flat_test.max()
print(f"\nBest test-set combo: {best_test_pair[0]} x {best_test_pair[1]} "
      f"-> Accuracy = {best_test_acc_val:.4f}")

test_matrix.to_csv(os.path.join(BASE_DIR, "data", "test_acc_matrix.csv"))

Voting Ensemble (ANN + SVM + LR) + Stacking

In [ ]:
import copy

# -- SMOTE for final models (AFTER CV) --
sm = SMOTE(k_neighbors=5, random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
X_res_df = pd.DataFrame(X_res, columns=feat_names)
y_res_s  = pd.Series(y_res)

# Fit 9 sklearn classifiers (exclude ANN — VotingClassifier will fit it fresh)
for name, clf in CLASSIFIERS.items():
    if name == "ANN":
        continue                              # defer — VotingClassifier fits it
    if name == "XGB":
        clf.fit(X_res_df, y_res_s, eval_set=[(X_val_df, y_val_s)], verbose=False)
    else:
        clf.fit(X_res, y_res)

print("8 sklearn classifiers fitted on SMOTE data. ANN deferred.")

# Ensure best_params_all["ANN"] is populated before use
if "ANN" not in best_params_all:
    print("Warning: Optuna tuning for ANN was interrupted. Using default ANN parameters.")
    best_params_all["ANN"] = {
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": 30
    }

# -- Voting = ANN + SVM + LR  (uses CLASSIFIERS directly) --
# Create a fresh ANNWrapper instance for the ensemble to avoid type recognition issues
# Using optimized parameters from Optuna (best_params_all["ANN"]) if available
ann_for_voting = ANNWrapper(
    n_features=X_train.shape[1],
    device_str=str(DEVICE),
    lr=best_params_all["ANN"]["lr"],
    weight_decay=best_params_all["ANN"]["weight_decay"],
    epochs=best_params_all["ANN"]["epochs"],
)
# Explicitly set _estimator_type as a defensive measure, though it should be set by the class
ann_for_voting._estimator_type = "classifier"

voting_clf = VotingClassifier(
    estimators=[
        ("ann", ann_for_voting),
        ("svm", CLASSIFIERS["SVM"]),
        ("lr",  CLASSIFIERS["LR"]),
    ],
    voting="soft",
    n_jobs=1,   # Avoid PyTorch pickling issues
)
voting_clf.fit(X_res, y_res)

y_pred_vote  = voting_clf.predict(X_test)
y_proba_vote = voting_clf.predict_proba(X_test)[:, 1]

print("\n--- Voting Ensemble (ANN + SVM + LR) ---")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_vote):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, y_proba_vote):.4f}")
print(classification_report(y_test, y_pred_vote,
                            target_names=["Benign", "Malignant"]))


# -- Stacking = SVM + RF + LR → XGBoost --
stacking_clf = StackingClassifier(
    estimators=[
        ("svm", CLASSIFIERS["SVM"]),
        ("rf",  CLASSIFIERS["RF"]),
        ("lr",  CLASSIFIERS["LR"]),
    ],
    final_estimator=XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3,
        scale_pos_weight=neg / pos, eval_metric="auc",
        random_state=42,
    ),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1,
)
stacking_clf.fit(X_res, y_res)

y_pred_stack  = stacking_clf.predict(X_test)
y_proba_stack = stacking_clf.predict_proba(X_test)[:, 1]

print("\n--- Stacking Ensemble (SVM + RF + LR → XGBoost) ---")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_stack):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_test, y_proba_stack):.4f}")
print(classification_report(y_test, y_pred_stack,
                            target_names=["Benign", "Malignant"]))

print("Ensembles fitted and evaluated.")

# -- Fit ANN separately (now that VotingClassifier is done with it) --
# The VotingClassifier's internal ANN (fresh clone) was fitted during voting_clf.fit().
# Now refit CLASSIFIERS["ANN"] on SMOTE so it can be used independently later.
CLASSIFIERS["ANN"].fit(X_res, y_res)
ann_fitted_val = CLASSIFIERS["ANN"].predict_proba(X_val)[:, 1]
print(f"ANN standalone fitted on SMOTE. Val AUC: {roc_auc_score(y_val, ann_fitted_val):.4f}")

Threshold Tuning

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"\nBest model from CV: {best_model_name}")

best_clf = CLASSIFIERS[best_model_name]
if best_model_name == "XGB":
    best_clf.fit(X_res_df, y_res_s, eval_set=[(X_val_df, y_val_s)], verbose=False)
else:
    best_clf.fit(X_res, y_res)

if best_model_name == "XGB":
    y_proba_val = best_clf.predict_proba(X_val_df)[:, 1]
else:
    y_proba_val = best_clf.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.1, 0.9, 81)
best_thresh, best_f1_val = 0.5, 0.0

print("\n--- Threshold Tuning ---")
for th in thresholds:
    f1_val = f1_score(y_val, (y_proba_val >= th).astype(int))
    if f1_val > best_f1_val:
        best_f1_val, best_thresh = f1_val, th

print(f"Best threshold: {best_thresh:.2f}  (F1={best_f1_val:.4f})")

if best_model_name == "XGB":
    y_proba_test = best_clf.predict_proba(X_test_df)[:, 1]
else:
    y_proba_test = best_clf.predict_proba(X_test)[:, 1]

y_pred_tuned = (y_proba_test >= best_thresh).astype(int)
print(f"Test performance at threshold {best_thresh:.2f}:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"  F1:       {f1_score(y_test, y_pred_tuned):.4f}")
print(f"  AUC-ROC:  {roc_auc_score(y_test, y_proba_test):.4f}")

# Plot
f1_scores = [f1_score(y_val, (y_proba_val >= t).astype(int)) for t in thresholds]
plt.figure(figsize=(8, 5))
plt.plot(thresholds, f1_scores, linewidth=2)
plt.axvline(best_thresh, color="red", linestyle="--", label=f"Best={best_thresh:.2f}")
plt.xlabel("Threshold"); plt.ylabel("F1 Score")
plt.title(f"Threshold Tuning — {best_model_name}")
plt.legend(); plt.tight_layout()
# Ensure the figures directory exists before saving
os.makedirs(os.path.join(BASE_DIR, "figures"), exist_ok=True)
plt.savefig(os.path.join(BASE_DIR, "figures", "threshold_tuning.png"), dpi=150)
plt.show()

Bootstrap 95% confidence Interval on best model

In [ ]:
print("\n--- Bootstrap 95% CI on Best Model ---")

N_BOOT = 1000
np.random.seed(42)

sm_ci = SMOTE(k_neighbors=5, random_state=42)
best_clf_ci = clone(CLASSIFIERS[best_model_name])
X_tr_ci, y_tr_ci = sm_ci.fit_resample(X_train, y_train)

if best_model_name == "XGB":
    best_clf_ci.set_params(early_stopping_rounds=None)
    best_clf_ci.fit(X_train_df, y_train_s)
    y_proba_ci = best_clf_ci.predict_proba(X_test_df)[:, 1]
else:
    best_clf_ci.fit(X_tr_ci, y_tr_ci)
    y_proba_ci = best_clf_ci.predict_proba(X_test)[:, 1]

y_pred_ci = (y_proba_ci >= 0.5).astype(int)

boot_accs, boot_aucs = [], []
for _ in range(N_BOOT):
    ix = np.random.choice(len(y_test), len(y_test), replace=True)
    boot_accs.append(accuracy_score(y_test[ix], y_pred_ci[ix]))
    boot_aucs.append(roc_auc_score(y_test[ix], y_proba_ci[ix]))

acc_m = np.mean(boot_accs)
acc_lo = np.percentile(boot_accs, 2.5)
acc_hi = np.percentile(boot_accs, 97.5)
auc_m  = np.mean(boot_aucs)
auc_lo = np.percentile(boot_aucs, 2.5)
auc_hi = np.percentile(boot_aucs, 97.5)

print(f"\n  {best_model_name} Bootstrap (n={N_BOOT}):")
print(f"  Accuracy: {acc_m:.4f}  [95% CI: {acc_lo:.4f} - {acc_hi:.4f}]")
print(f"  AUC-ROC:  {auc_m:.4f}  [95% CI: {auc_lo:.4f} - {auc_hi:.4f}]")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(boot_accs, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
ax1.axvline(acc_lo, color="red", ls="--", label=f"2.5%: {acc_lo:.4f}")
ax1.axvline(acc_hi, color="red", ls="--", label=f"97.5%: {acc_hi:.4f}")
ax1.axvline(acc_m, color="black", lw=2, label=f"Mean: {acc_m:.4f}")
ax1.set_title(f"Bootstrap Accuracy - {best_model_name}")
ax1.set_xlabel("Accuracy"); ax1.legend(fontsize=8)

ax2.hist(boot_aucs, bins=40, color="darkorange", edgecolor="white", alpha=0.8)
ax2.axvline(auc_lo, color="red", ls="--", label=f"2.5%: {auc_lo:.4f}")
ax2.axvline(auc_hi, color="red", ls="--", label=f"97.5%: {auc_hi:.4f}")
ax2.axvline(auc_m, color="black", lw=2, label=f"Mean: {auc_m:.4f}")
ax2.set_title(f"Bootstrap AUC-ROC - {best_model_name}")
ax2.set_xlabel("AUC-ROC"); ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "bootstrap_ci.png"), dpi=150)
plt.show()
print("Saved: bootstrap_ci.png")

Ablation Study

In [ ]:
print("\n--- Ablation Study: Component Contribution ---")

baseline_acc = results_df.iloc[0]["CV_Acc_Mean"]

ablation_results = []

# 1. Full pipeline (baseline)
ablation_results.append(("Full pipeline (SMOTE + Optuna + FS)", baseline_acc, "-"))

# 2. Remove SMOTE — fit on imbalanced X_train directly
try:
    best_clf_no_smote = clone(CLASSIFIERS[best_model_name])
    if best_model_name == "XGB":
        best_clf_no_smote.set_params(early_stopping_rounds=None)
        best_clf_no_smote.fit(X_train_df, y_train_s)
        acc_no_smote = accuracy_score(y_test,
            best_clf_no_smote.predict(X_test_df))
    else:
        best_clf_no_smote.fit(X_train, y_train)
        acc_no_smote = accuracy_score(y_test,
            best_clf_no_smote.predict(X_test))
    drop_smote = baseline_acc - acc_no_smote
    ablation_results.append(("Remove SMOTE", acc_no_smote,
                             f"drop={drop_smote:.4f}"))
except Exception:
    ablation_results.append(("Remove SMOTE", np.nan, "ERR"))

# 3. Remove Optuna — use default hyperparams
try:
    best_clf_default = clone(CLASSIFIERS_DEFAULTS.get(
        best_model_name, CLASSIFIERS[best_model_name]))
    if best_model_name == "XGB":
        best_clf_default.set_params(early_stopping_rounds=None)
        best_clf_default.fit(X_res_df, y_res_s)
        acc_default = accuracy_score(y_test,
            best_clf_default.predict(X_test_df))
    else:
        best_clf_default.fit(X_res, y_res)
        acc_default = accuracy_score(y_test,
            best_clf_default.predict(X_test))
    drop_optuna = baseline_acc - acc_default
    ablation_results.append(("Remove Optuna (default params)", acc_default,
                             f"drop={drop_optuna:.4f}"))
except Exception:
    ablation_results.append(("Remove Optuna", np.nan, "ERR"))

# 4. Remove FS — use all 9 features only
try:
    best_clf_allfeat = clone(CLASSIFIERS[best_model_name])
    if best_model_name == "XGB":
        best_clf_allfeat.set_params(early_stopping_rounds=None)
        best_clf_allfeat.fit(X_res_df, y_res_s)
        acc_allfeat = accuracy_score(y_test,
            best_clf_allfeat.predict(X_test_df))
    else:
        best_clf_allfeat.fit(X_res, y_res)
        acc_allfeat = accuracy_score(y_test,
            best_clf_allfeat.predict(X_test))
    drop_fs = baseline_acc - acc_allfeat
    ablation_results.append(("Remove FS (All 9 features)", acc_allfeat,
                             f"drop={drop_fs:.4f}"))
except Exception:
    ablation_results.append(("Remove FS", np.nan, "ERR"))

ablation_df = pd.DataFrame(ablation_results,
                           columns=["Configuration", "Accuracy", "Impact"])

print("\n" + "=" * 70)
print("   ABLATION STUDY — What happens when you remove each component?")
print("=" * 70)
print(ablation_df.to_string(index=False))
print(f"\n  Each 'drop' value is accuracy lost vs full pipeline ({baseline_acc:.4f}).")
print(f"  Larger drop = component is more important.")
print("=" * 70)

**SHAP**

In [ ]:
print("Computing SHAP values ...")
explainer = shap.TreeExplainer(CLASSIFIERS["XGB"])
shap_values = explainer.shap_values(X_test_orig_df)
print(f"SHAP values shape: {shap_values.shape}")

plt.figure()
shap.summary_plot(shap_values, X_test_orig_df, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "shap_bar.png"), dpi=150)
plt.show()

plt.figure()
shap.summary_plot(shap_values, X_test_orig_df, show=False)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "shap_beeswarm.png"), dpi=150)
plt.show()

plt.figure()
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_test_orig_df.iloc[0].values,
        feature_names=feat_names.tolist(),
    ),
    show=False,
)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "shap_waterfall.png"), dpi=150)
plt.show()

print("SHAP analysis complete.")

XGBoost fit + evaluation

In [ ]:
xgb_final = XGBClassifier(
    n_estimators=best_params_all["XGB"]["n_estimators"],
    learning_rate=best_params_all["XGB"]["learning_rate"],
    max_depth=best_params_all["XGB"]["max_depth"],
    subsample=best_params_all["XGB"]["subsample"],
    colsample_bytree=best_params_all["XGB"]["colsample_bytree"],
    scale_pos_weight=neg / pos, eval_metric="auc", random_state=42,
    early_stopping_rounds=20,
)
xgb_final.fit(X_res_df, y_res_s, eval_set=[(X_val_df, y_val_s)], verbose=False)

y_pred_final  = xgb_final.predict(X_test_df)
y_proba_final = xgb_final.predict_proba(X_test_df)[:, 1]

print("\n--- XGBoost Final Evaluation ---")
print(classification_report(y_test_s, y_pred_final,
                            target_names=["Benign", "Malignant"]))
print(f"Accuracy    : {accuracy_score(y_test_s, y_pred_final):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_test_s, y_proba_final):.4f}")

cm = confusion_matrix(y_test_s, y_pred_final)
tn, fp, fn, tp = cm.ravel()
sens = tp / (tp + fn) if (tp + fn) > 0 else 0
spec = tn / (tn + fp) if (tn + fp) > 0 else 0
print(f"Sensitivity : {sens:.4f}")
print(f"Specificity : {spec:.4f}")

RocCurveDisplay.from_predictions(y_test_s, y_proba_final)
plt.title("ROC — XGBoost Breast Cancer")
plt.savefig(os.path.join(BASE_DIR, "figures", "roc_xgboost.png"), dpi=150)
plt.show()

CLASSIFIERS["XGB"] = xgb_final

Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
PrecisionRecallDisplay.from_estimator(xgb_final, X_test_df, y_test_s, ax=ax)
plt.title("Precision-Recall Curve — XGBoost")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "pr_curve.png"), dpi=150)
plt.show()
print("PR curve saved.")

Confusion Matices ( All classifiers)

In [ ]:
N_MODELS = len(CLASSIFIERS)   # 10
ncols = 5
nrows = 2
fig, axes = plt.subplots(nrows, ncols, figsize=(22, 9))
axes = axes.flatten()

for ax_i, (name, clf) in zip(axes, CLASSIFIERS.items()):
    if name == "XGB":
        y_pred_i = clf.predict(X_test_df)
    else:
        y_pred_i = clf.predict(X_test)
    cm_i = confusion_matrix(y_test, y_pred_i)
    ConfusionMatrixDisplay(cm_i, display_labels=["Benign", "Malignant"]).plot(
        ax=ax_i, cmap=plt.cm.Blues, colorbar=False,
    )
    ax_i.set_title(name)

plt.suptitle("Confusion Matrices — All 10 Classifiers", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "confusion_all_10_models.png"),
            dpi=150, bbox_inches="tight")
plt.show()

ROC all models

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(11, 8))
colors_10 = plt.cm.tab10(np.linspace(0, 1, 10))

for (name, clf), color in zip(CLASSIFIERS.items(), colors_10):
    y_proba = clf.predict_proba(
        X_test_df if name == "XGB" else X_test
    )[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})",
             color=color, linewidth=2)

# Voting
fpr_v, tpr_v, _ = roc_curve(y_test, y_proba_vote)
plt.plot(fpr_v, tpr_v,
         label=f"Voting ANN+SVM+LR (AUC={roc_auc_score(y_test, y_proba_vote):.4f})",
         color="black", linewidth=2.5, linestyle="--")

# Stacking
fpr_s, tpr_s, _ = roc_curve(y_test, y_proba_stack)
plt.plot(fpr_s, tpr_s,
         label=f"Stacking (AUC={roc_auc_score(y_test, y_proba_stack):.4f})",
         color="red", linewidth=2.5, linestyle="--")

plt.plot([0, 1], [0, 1], "k:", linewidth=1, label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — All 10 Models")
plt.legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "roc_all_models.png"), dpi=150)
plt.show()

Learning Curves

In [ ]:
print("\n--- Learning Curves ---")

best_clf_lc = CLASSIFIERS[best_model_name]
pipe_lc = ImbPipeline([
    ("sm", SMOTE(k_neighbors=5, random_state=42)),
    ("clf", best_clf_lc),
])

train_sizes, train_scores, val_scores = learning_curve(
    pipe_lc,
    X_train_df if best_model_name == "XGB" else X_train,
    y_train_s if best_model_name == "XGB" else y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring="roc_auc", n_jobs=-1, random_state=42,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(8, 6))
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.2, color="blue")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                 alpha=0.2, color="orange")
plt.plot(train_sizes, train_mean, "o-", color="blue", label="Training AUC")
plt.plot(train_sizes, val_mean, "o-", color="orange", label="Validation AUC")
plt.xlabel("Training samples"); plt.ylabel("AUC")
plt.title(f"Learning Curve — {best_model_name}")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "learning_curve.png"), dpi=150)
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"Learning curve saved. Gap: {gap:.4f} "
      f"({'low overfitting' if gap < 0.05 else 'possible overfitting'})")


Calibration Curves

In [ ]:
if 'colors_10' not in dir():
    colors_10 = plt.cm.tab10(np.linspace(0, 1, 10))
print("\n--- Calibration Curves ---")

plt.figure(figsize=(11, 8))
for (name, clf), color in zip(CLASSIFIERS.items(), colors_10):
    y_proba = clf.predict_proba(
        X_test_df if name == "XGB" else X_test
    )[:, 1]
    prob_true, prob_pred = calibration_curve(
        y_test, y_proba, n_bins=10, strategy="uniform",
    )
    plt.plot(prob_pred, prob_true, marker="o", label=name,
             color=color, linewidth=2)

plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Calibration Curves — All 10 Models")
plt.legend(loc="lower right", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "calibration_curves.png"), dpi=150)
plt.show()
print("Calibration curves saved.")

McNemar's Statistical Test

In [ ]:
print("\n--- McNemar's Test ---")
from scipy.stats import chi2


def mcnemar_test(y_true, y_pred_a, y_pred_b):
    a_correct = y_pred_a == y_true
    b_correct = y_pred_b == y_true
    b = np.sum(a_correct & ~b_correct)
    c = np.sum(~a_correct & b_correct)
    if b + c == 0:
        return 1.0
    chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
    return 1 - chi2.cdf(chi2_stat, 1)


best_name = results_df.iloc[0]["Model"]
best_clf_mc = CLASSIFIERS[best_name]
y_pred_best = (best_clf_mc.predict(X_test_df)
               if best_name == "XGB"
               else best_clf_mc.predict(X_test))

mcnemar_rows = []
for name, clf in CLASSIFIERS.items():
    if name == best_name:
        continue
    y_pred_other = (clf.predict(X_test_df)
                    if name == "XGB"
                    else clf.predict(X_test))
    p = mcnemar_test(y_test, y_pred_best, y_pred_other)
    mcnemar_rows.append((name, p,
                         "SIGNIFICANT" if p < 0.05 else "not significant"))

mcnemar_df = pd.DataFrame(mcnemar_rows, columns=["Model", "p-value", "vs Best"])
print(f"\nMcNemar's test: {best_name} vs all others")
print(mcnemar_df.to_string(index=False))

p_vote = mcnemar_test(y_test, y_pred_best, voting_clf.predict(X_test))
print(f"\n{best_name} vs Voting: p={p_vote:.6f} "
      f"— {'SIGNIFICANT' if p_vote < 0.05 else 'not significant'}")

p_stack = mcnemar_test(y_test, y_pred_best, stacking_clf.predict(X_test))
print(f"{best_name} vs Stacking: p={p_stack:.6f} "
      f"— {'SIGNIFICANT' if p_stack < 0.05 else 'not significant'}")


Feature importance RF + XGBoost

In [ ]:
print("\n--- Feature Importance: RF vs XGBoost ---")

rf_imp  = CLASSIFIERS["RF"].feature_importances_
xgb_imp = CLASSIFIERS["XGB"].feature_importances_

imp_df = pd.DataFrame({
    "Feature": feat_names,
    "RF": rf_imp,
    "XGBoost": xgb_imp,
}).sort_values("RF", ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.barh(imp_df["Feature"], imp_df["RF"], color="steelblue")
ax1.set_title("Random Forest")
ax1.set_xlabel("Importance")
ax2.barh(imp_df["Feature"], imp_df["XGBoost"], color="darkorange")
ax2.set_title("XGBoost")
ax2.set_xlabel("Importance")
plt.suptitle("Feature Importance — RF vs XGBoost", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "figures", "feature_importance_comparison.png"),
            dpi=150)
plt.show()
print("Feature importance comparison saved.")

BreastANN full training

In [ ]:
# Report final ANN performance on val and test using the already-fitted model.
ann_fitted = CLASSIFIERS["ANN"]

ann_model = ann_fitted.model          # underlying PyTorch model
if ann_model is not None:
    ann_model.eval()
    device_ann = torch.device(ann_fitted.device_str)

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device_ann)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).to(device_ann)
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device_ann)

    with torch.no_grad():
        val_out   = ann_model(X_val_t)
        val_auc   = roc_auc_score(y_val, torch.sigmoid(val_out).cpu().numpy())
        test_out  = ann_model(X_test_t)
        test_auc  = roc_auc_score(y_test, torch.sigmoid(test_out).cpu().numpy())

    print("\n--- ANN Validation Report ---")
    print(f"  Val AUC : {val_auc:.4f}")
    print(f"  Test AUC: {test_auc:.4f}")
    print(f"  Params  : lr={ann_fitted.lr}, "
          f"weight_decay={ann_fitted.weight_decay}, "
          f"epochs={ann_fitted.epochs}")
else:
    print("\nANN model is None — was it fitted?")

EfficientNet

In [ ]:
print("""
┌──────────────────────────────────────────────────────────┐
│  EFFICIENTNET IMAGE BRANCH — PLACEHOLDER                 │
│  Requires a real dataset (BreakHis, CBIS-DDSM, MIAS).    │
│  NOT connected to the WBCD classification pipeline.      │
└──────────────────────────────────────────────────────────┘
""")

backbone = models.efficientnet_b0(pretrained=True)
for p in backbone.parameters():
    p.requires_grad = False
in_feats = backbone.classifier[1].in_features
backbone.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_feats, 2))
backbone.to(DEVICE).eval()
print("EfficientNet-B0 backbone initialised (frozen).")

U-Net

In [ ]:
print("""
┌──────────────────────────────────────────────────────────┐
│  U-NET SEGMENTATION — PLACEHOLDER                        │
│  For zero-shot segmentation, consider SAM.               │
└──────────────────────────────────────────────────────────┘
""")

Model Save + ONNX export

In [ ]:
import os
import sys

# Define BASE_DIR if it's not already defined
if 'BASE_DIR' not in locals() and 'BASE_DIR' not in globals():
    IN_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_GPU", "")
    BASE_DIR = (
        "/content/drive/MyDrive/BC_internship"
        if IN_COLAB
        else os.path.join(os.getcwd(), "BC_internship")
    )

SAVE_DIR = os.path.join(BASE_DIR, "src", "models")
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(xgb_final,     os.path.join(SAVE_DIR, "xgb_breast.pkl"))
joblib.dump(scaler,        os.path.join(SAVE_DIR, "scaler.pkl"))
joblib.dump(voting_clf,    os.path.join(SAVE_DIR, "voting_ensemble.pkl"))
joblib.dump(stacking_clf,  os.path.join(SAVE_DIR, "stacking_ensemble.pkl"))
# Save the full CLASSIFIERS dict (sklearn-compatible objects)
joblib.dump(CLASSIFIERS,   os.path.join(SAVE_DIR, "all_10_classifiers.pkl"))
print("Saved: xgb_breast.pkl, scaler.pkl, voting_ensemble.pkl, "
      "stacking_ensemble.pkl, all_10_classifiers.pkl")

# Save ANN state dict separately (for portability)
if ann_model is not None:
    torch.save(ann_model.state_dict(),
               os.path.join(SAVE_DIR, "breast_ann.pth"))
    m2 = BreastANN(n_features=X_train.shape[1])
    m2.load_state_dict(torch.load(
        os.path.join(SAVE_DIR, "breast_ann.pth"), map_location=DEVICE,
    ))
    print("Saved & verified: breast_ann.pth")

dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
torch.onnx.export(
    backbone, dummy,
    os.path.join(SAVE_DIR, "efficientnet_breast.onnx"),
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}},
    opset_version=17,
)
print("Saved: efficientnet_breast.onnx")

if ann_model is not None:
    q_model = torch.quantization.quantize_dynamic(
        ann_model.to("cpu"), {nn.Linear}, dtype=torch.qint8,
    )
    torch.save(q_model.state_dict(),
               os.path.join(SAVE_DIR, "breast_ann_int8.pth"))
    print("Saved: breast_ann_int8.pth")

with open(os.path.join(SAVE_DIR, "best_params.json"), "w") as f:
    json.dump(best_params_all, f, indent=2)

print("\nAll models exported successfully.")

Medical report


In [ ]:
from PIL import Image
from torchvision import transforms
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import numpy as np # Import numpy for array creation

# Assuming EfficientNet has been trained and saved with its weights
# For demonstration, we'll use the 'backbone' defined earlier.
# In a real scenario, you would load a *trained* EfficientNet for mammograms.

def preprocess_image(image_obj):
    """
    Preprocesses a mammogram PIL Image for EfficientNet.
    """
    # Define the same transformations used during EfficientNet training
    # These are typical for ImageNet-pretrained models
    preprocess = transforms.Compose([
        transforms.Resize(256),            # Resize to 256 for cropping
        transforms.CenterCrop(224),        # Crop to 224x224
        transforms.Grayscale(num_output_channels=3), # Mammograms are grayscale, but EfficientNet expects 3 channels
        transforms.ToTensor(),             # Convert to PyTorch Tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # ImageNet stats
    ])

    try:
        # Ensure image_obj is already a PIL Image, convert to L for grayscale then to RGB
        image_tensor = preprocess(image_obj.convert('RGB'))
        return image_tensor.unsqueeze(0) # Add batch dimension
    except Exception as e:
        print(f"Error processing image: {e}")
        return None

def medical_report(tabular_data=None, image_obj=None, output_widget=None):
    """
    Unified medical inference using all 10 fitted classifiers and EfficientNet.

    Parameters
    ----------
    tabular_data : array-like, optional
        Raw (unscaled) features for one patient, length = n_features.
        Required if 'Tabular' input is selected.
    image_obj : PIL.Image or None, optional
        A PIL Image object of the mammogram. Required if 'Image' input is selected.
    output_widget : ipywidgets.Output or None, optional
        An ipywidgets.Output widget to direct print statements to.

    Returns
    -------
    dict with keys: scores, combined, risk
    """
    scores = {}
    numeric_scores = []

    # Tabular data processing
    if tabular_data is not None and len(tabular_data) == len(feat_names):
        x_scaled = scaler.transform([tabular_data])
        x_scaled_df = pd.DataFrame(x_scaled, columns=feat_names)

        # All 10 tabular classifiers  (ANN.predict_proba works via ANNWrapper)
        for name, clf in CLASSIFIERS.items():
            try:
                scores[name] = clf.predict_proba(
                    x_scaled_df if name == "XGB" else x_scaled
                )[0, 1]
            except Exception as e:
                scores[name] = f"ERR: {e}"

        # Voting ensemble
        try:
            scores["Voting"] = voting_clf.predict_proba(x_scaled)[0, 1]
        except Exception:
            scores["Voting"] = "N/A"

        # Stacking ensemble
        try:
            scores["Stacking"] = stacking_clf.predict_proba(x_scaled)[0, 1]
        except Exception:
            scores["Stacking"] = "N/A"

        numeric_scores.extend([v for v in scores.values() if isinstance(v, (int, float))])

    # EfficientNet (Image Branch)
    image_prediction_score = None
    if image_obj:
        try:
            image_tensor = preprocess_image(image_obj)
            if image_tensor is not None:
                backbone.eval() # Set model to evaluation mode
                with torch.no_grad():
                    image_output = backbone(image_tensor.to(DEVICE)) # Pass image through model
                    # EfficientNet output is logits, convert to probability for probability
                    image_prediction_score = torch.sigmoid(image_output).cpu().numpy()[0, 1]
                    scores["EfficientNet"] = image_prediction_score
                    numeric_scores.append(image_prediction_score)
        except Exception as e:
            if output_widget:
                with output_widget:
                    print(f"Error during EfficientNet prediction: {e}")
            scores["EfficientNet"] = f"ERR: {e}"

    combined = np.mean(numeric_scores) if numeric_scores else 0.0

    risk = ("HIGH" if combined > 0.7
            else "MODERATE" if combined > 0.3
            else "LOW")

    if output_widget:
        with output_widget:
            print("\n" + "=" * 55)
            print("         BREAST CANCER ANALYSIS REPORT")
            print("=" * 55)
            if not scores:
                print("  No data provided for analysis.")
            for name, proba in scores.items():
                if isinstance(proba, (int, float)):
                    print(f"  {name:>20s}: {proba:.2%}")
                else:
                    print(f"  {name:>20s}: {proba}")
            print("-" * 55)
            print(f"  {'Combined (mean)':>20s}: {combined:.2%}")
            print(f"  {'ASSESSMENT':>20s}: {risk} RISK")
            print("=" * 55 + "\n")
    else:
        print("\n" + "=" * 55)
        print("         BREAST CANCER ANALYSIS REPORT")
        print("=" * 55)
        if not scores:
            print("  No data provided for analysis.")
        for name, proba in scores.items():
            if isinstance(proba, (int, float)):
                print(f"  {name:>20s}: {proba:.2%}")
            else:
                print(f"  {name:>20s}: {proba}")
        print("-" * 55)
        print(f"  {'Combined (mean)':>20s}: {combined:.2%}")
        print(f"  {'ASSESSMENT':>20s}: {risk} RISK")
        print("=" * 55 + "\n")

    return {"scores": scores, "combined": combined, "risk": risk}


def interactive_inference():
    """
    Prompts the user for input type and data, then runs the medical report.
    """
    print("\n" + "=" * 60)
    print("  BREAST CANCER RISK ASSESSMENT \u2014 Select Input Type")
    print("=" * 60)
    print("Please make a selection")

    input_type_selector = widgets.RadioButtons(
        options=['Tabular only', 'Image only'],
        description='Input Type:',
        disabled=False,
        value=None  # Explicitly set to None for no initial selection
    )

    output_widget = widgets.Output() # For general messages like "Selected: Tabular only"
    dynamic_input_area = widgets.Output() # New widget for displaying input fields

    file_upload_widget = widgets.FileUpload(
        accept='image/*',  # Accepted file type
        multiple=False     # Only one file at a time
    )

    # Store references to input widgets for later access
    tabular_input_widgets = []
    submit_button = widgets.Button(description="Generate Report")
    reset_button = widgets.Button(description="Reset") # New reset button
    report_output = widgets.Output() # This is for the final medical report

    global uploaded_image_pil # Declare as global to store the image for later
    uploaded_image_pil = None
    global input_choice # Global to store the selected input type
    input_choice = None

    def on_upload_change(change):
        global uploaded_image_pil
        if file_upload_widget.value:
            uploaded_filename = list(file_upload_widget.value.keys())[0]
            uploaded_content = file_upload_widget.value[uploaded_filename]['content']

            try:
                uploaded_image_pil = Image.open(io.BytesIO(uploaded_content))
                with output_widget: # Messages go here
                    print(f"Successfully uploaded {uploaded_filename}.")
            except Exception as e:
                uploaded_image_pil = None
                with output_widget: # Messages goes here
                    print(f"Error processing image file: {e}")
        else:
            uploaded_image_pil = None
            with output_widget: # Messages go here
                print("No file uploaded.")

    file_upload_widget.observe(on_upload_change, names='value')

    # Display the selector and the two output areas
    display(input_type_selector, output_widget, dynamic_input_area)

    # Remove redundant initial message, it will be handled by on_input_type_change
    # with dynamic_input_area:
    #     print("Please select an input type from above to proceed.")

    def on_input_type_change(change):
        global input_choice
        global uploaded_image_pil
        with output_widget:
            clear_output() # Clear previous "Selected:" message
            input_choice = change['new'] # Access 'new' using bracket notation
            if input_choice: # Only print if a selection is made (not None)
                print(f"Selected: {input_choice}")

        with dynamic_input_area: # Now clear and display input fields here
            clear_output() # Clear previous input fields (e.g., if switching from image to tabular)
            # Clear previous tabular input widgets and reset uploaded image
            tabular_input_widgets.clear()
            uploaded_image_pil = None # Reset image on input type change

            # Clear file upload widget
            file_upload_widget.unobserve(on_upload_change, names='value')
            file_upload_widget._counter = 0 # Reset the counter to clear the widget
            file_upload_widget.observe(on_upload_change, names='value')
            with output_widget:
                print("File upload widget cleared.")

            # --- Display input fields based on selection ---
            input_elements = [] # List to hold widgets for dynamic display

            if input_choice == 'Tabular only':
                input_elements.append(widgets.Label("Please provide tabular data (all fields are required):"))

                for i, nm in enumerate(feat_names):
                    # input_field = widgets.Text(
                    #     description=f'{nm}:',
                    #     continuous_update=False,
                    #     layout=widgets.Layout(width='auto', description_width='initial')
                    # )
                    input_field = widgets.Text(
                        description=f'{nm}:',
                        continuous_update=False,
                        style={'description_width': '250px'},
                        layout=widgets.Layout(width='450px')
                    )
                    tabular_input_widgets.append(input_field)
                    input_elements.append(input_field)
                input_elements.append(widgets.HBox([submit_button, reset_button]))
                input_elements.append(report_output)
                display(widgets.VBox(input_elements)) # Display in dynamic_input_area

            elif input_choice == 'Image only':
                image_elements = []
                image_elements.append(widgets.Label("Please upload an image:"))

                image_elements.append(file_upload_widget)
                image_elements.append(widgets.HBox([submit_button, reset_button]))
                image_elements.append(report_output)
                display(widgets.VBox(image_elements)) # Display in dynamic_input_area
            else: # If input_choice is None (after reset)
                print("Please select an input type from above to proceed.")

            # The general display of buttons and report_output is removed from here

    # Manually trigger the initial rendering for 'Tabular only' is removed.
    input_type_selector.observe(on_input_type_change, names='value')

    # Explicitly set value to None *after* observer is attached to ensure initial state is correctly processed
    input_type_selector.value = None

    def on_submit_button_click(b):
        with report_output:
            clear_output()
            current_tabular_data = None
            current_image_obj = None

            if input_choice == 'Tabular only':
                try:
                    current_tabular_data = []
                    all_fields_filled = True
                    for i, widget in enumerate(tabular_input_widgets):
                        if widget.value.strip(): # Check if the text input is not empty
                            current_tabular_data.append(float(widget.value))
                        else:
                            print(f"Error: '{feat_names[i]}' field is empty. All tabular fields must be filled.")
                            all_fields_filled = False
                            break

                    if not all_fields_filled:
                        return

                    current_tabular_data = np.array(current_tabular_data)
                except ValueError:
                    print("Error: Invalid input for tabular data. Please enter numbers.")
                    return
                except Exception as e:
                    print(f"Error collecting tabular data: {e}")
                    return
            elif input_choice == 'Image only':
                # For 'Image only', tabular_data is optional for medical_report,
                # but we'll pass mean values to allow combined assessment if an image is provided.
                current_tabular_data = np.array([X_train_orig[:, i].mean() for i in range(len(feat_names))])

            if input_choice == 'Image only': # Only check image for 'Image only' type
                if uploaded_image_pil:
                    current_image_obj = uploaded_image_pil
                else:
                    print("Error: Image input required but no image uploaded.")
                    return

            if current_tabular_data is None and current_image_obj is None:
                print("Error: No valid input data provided for analysis.")
                return

            # Call medical_report with collected data
            medical_report(tabular_data=current_tabular_data, image_obj=current_image_obj, output_widget=report_output)

    def on_reset_button_click(b):
        global input_choice # Make sure input_choice can be reset
        with report_output:
            clear_output() # Clear the report output
        with output_widget:
            clear_output() # Clear any messages in the main output_widget

        with dynamic_input_area: # Clear the dynamic input area as well
            clear_output() # Clear any input fields displayed here
            # Reset file upload widget
            global uploaded_image_pil
            uploaded_image_pil = None
            file_upload_widget.unobserve(on_upload_change, names='value')
            file_upload_widget._counter = 0 # Reset the counter to clear the widget
            file_upload_widget.observe(on_upload_change, names='value')
            with output_widget:
                print("File upload widget cleared.")

            # Clear tabular input fields
            for widget in tabular_input_widgets:
                widget.value = ''

            # Set input_type_selector to None to effectively unselect it
            input_type_selector.value = None

    submit_button.on_click(on_submit_button_click)
    reset_button.on_click(on_reset_button_click) # Attach reset button handler


# Ensure best_fs is defined (in case previous cells weren't run sequentially)
if 'fs_df' not in locals() and 'fs_df' not in globals():
    # Fallback if fs_df is not defined (e.g., if only this cell is run)
    best_fs = {'Method': 'N/A', 'CV_AUC': 0.0} # Placeholder
    print("Warning: fs_df not found. Using placeholder for best_fs.")

# This block should be after the definition of interactive_inference
# and any other necessary functions/variables for the initial printouts.
print("\n" + "=" * 60)
print("  PIPELINE COMPLETE \u2014 10 classifiers")
print("=" * 60)
print(f"  Best model   : {best_model_name} "
      f"(CV AUC={results_df.iloc[0]['CV_AUC_Mean']:.4f})")
print(f"  Best FS      : {best_fs['Method']} "
      f"(AUC={best_fs['CV_AUC']:.4f})")
print(f"  Figures saved: {os.path.join(BASE_DIR, 'figures')}")
print(f"  Models saved : {os.path.join(BASE_DIR, 'src', 'models')}")
print("=" * 60)

interactive_inference()

Overall Update

In [ ]:
print("=== UPDATE ===")
print(f"Best model: {best_model_name}")
print(f"Best CV AUC: {results_df.iloc[0]['CV_AUC_Mean']:.4f}")
print(f"Best CV Acc: {results_df.iloc[0]['CV_Acc_Mean']:.4f}")
print(f"Voting AUC: {roc_auc_score(y_test, y_proba_vote):.4f}")
print(f"Stacking AUC: {roc_auc_score(y_test, y_proba_stack):.4f}")
print(f"Best FS method: {best_fs['Method']} — {best_fs['N_Features']} features")
print(f"PSO AUC: {pso_mask.sum()} features selected")
print(f"WOA AUC: {woa_mask.sum()} features selected")
print(f"Best threshold: {best_thresh:.2f}")